**_querying from source_**

In [0]:
%sql
select * from pyspark_data.source.products

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

### **_reading sql table into dataframe & deduplication_**

In [0]:
df  = spark.sql("select * from pyspark_data.source.products")

## adding dedup column
df = df.withColumn("dedup",row_number().over(Window.partitionBy("id").orderBy(desc("updatedDate"))))
df = df.filter(df.dedup == 1).drop("dedup")
display(df)

### # **_upserts_**

In [0]:
from delta.tables import DeltaTable

## using try catch to avoid errors in case if table is not there else creating it
if len(dbutils.fs.ls("/Volumes/pyspark_data/source/dbvolume/products_sink/")) > 0:
    print("upsert in progress")
    dlt_obj = DeltaTable.forPath(spark, "/Volumes/pyspark_data/source/dbvolume/products_sink/")
    dlt_obj.alias("tgt").merge(
        df.alias("src"),
        "src.id = tgt.id"
    ).whenMatchedUpdateAll(
        condition="src.updatedDate >= tgt.updatedDate"
    ).whenNotMatchedInsertAll().execute()
    print("Upsert in done")

else:
    print(f"Error: {e}")
    df.write.format("delta").mode("overwrite").save("/Volumes/pyspark_data/source/dbvolume/products_sink/")

### **_querying from products sink_**

In [0]:
%sql
select * from delta.`/Volumes/pyspark_data/source/dbvolume/products_sink`